In [1]:
import os, sys
sys.path.append(os.path.abspath('..'))

In [2]:
# Library imports
import json
import time
import numpy as np
import pandas as pd
import torch
import gymnasium as gym

from env import InventoryEnv
from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker

# Same normalization constant env.py's get_state() already uses for warehouses_distance
# (max warehouse-to-customer distance for the placements in env.py, e.g. (-50,-50) to a
# (100,100) corner is sqrt(150**2 + 150**2)). Reused here to rescale the reward fed to
# the learner - NOT applied inside env.py itself, since env.step()'s raw distance reward
# is also the evaluation metric reported throughout export_results.ipynb (real total
# distance, matching the paper's Eq. 9 units); rescaling it there would silently change
# the units of every results table. A uniform positive rescale of every step's reward
# doesn't change the optimal policy (trajectory ordering, and hence the argmax policy,
# is preserved under multiplication by a positive constant) - it just keeps returns
# closer to the O(1) scale PPO's critic and DQN's Q-regression assume under
# stable-baselines3's default hyperparameters (learning rate, network init), instead of
# the O(1e5) raw cumulative distance for the larger instances trained here.
DISTANCE_NORM = 212.13


class ScaledRewardWrapper(gym.RewardWrapper):
    """Divides every step's reward by DISTANCE_NORM before the learning algorithm sees
    it. See DISTANCE_NORM's comment for why this is done here instead of in env.py."""

    def reward(self, reward):
        return reward / DISTANCE_NORM


def _ppo_action_mask(env):
    """Passed to ActionMasker - matches PPOPolicy._rl_get_state's masking logic in
    policies/ppo_policy.py (the mask MaskablePPO's predict() is given at inference time)
    so training and inference agree on which warehouses are eligible."""
    return np.array([capacity > 0 for capacity in env.warehouses_capacity])

In [3]:
%load_ext tensorboard
%tensorboard --logdir='proximal_policy_optimization_training/Data_Training' --port 6006    #Change port if needed (6006,9009,9999)
#pip install --upgrade tensorboard tensorflow

In [4]:
def train_ppo(num_warehouses, num_customers, capacity_distribution):

    # Deliberate reseed. np.random.seed also covers env.create_customer() (numpy's
    # *global* RNG, since seed= passed to MaskablePPO() below only seeds SB3's own
    # internals).
    SEED = 42
    np.random.seed(SEED)

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
    env = ActionMasker(env, _ppo_action_mask)
    env = ScaledRewardWrapper(env)

    env.reset()

    model_PPO = MaskablePPO(
        "MlpPolicy",
        env,
        tensorboard_log=f"proximal_policy_optimization_training/Data_Training/PPO/w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}/",
        verbose=0,
        seed=SEED,  # seeds SB3's own internals (policy init, action sampling, rollout buffer)
    )

    # If this ever moves to a vectorized env (e.g. SubprocVecEnv) for speed, each
    # subprocess needs its own derived seed - child processes don't inherit the
    # parent's already-advanced RNG state deterministically.

    n_steps = num_customers * 50_000    #10_000

    start_time = time.perf_counter()
    model_PPO.learn(n_steps, reset_num_timesteps=True)
    training_time_minutes = (time.perf_counter() - start_time) / 60

    model_PPO.save(f'proximal_policy_optimization_training/rl_models/ppo_models/ppo_model_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.zip')
    print('Saved PPO model')

    return training_time_minutes

# NOTE: even with the above seeded, exact reproducibility across machines/time also
# needs pinned package versions (torch, stable-baselines3, sb3-contrib, CUDA/cuDNN -
# requirements.txt already pins the Python package versions) - not addressed here.

In [5]:
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
num_warehouses_options = [5]
num_customers_options = [400]
capacity_distribution_options = ['uneven']

training_times = []  # one row per (num_warehouses, num_customers, capacity_distribution) family

for num_customers in num_customers_options:
    for num_warehouses in num_warehouses_options:
        for capacity_distribution in capacity_distribution_options:
            training_time_minutes = train_ppo(num_warehouses, num_customers, capacity_distribution)
            print(f"Trained model for {num_warehouses} warehouses, {num_customers} customers, {capacity_distribution} capacity")

            training_times.append({
                'num_warehouses': num_warehouses,
                'num_customers': num_customers,
                'capacity_distribution': capacity_distribution,
                'training_time': round(training_time_minutes, 2),
            })

info_dir = 'proximal_policy_optimization_training'
os.makedirs(info_dir, exist_ok=True)

training_times_df = pd.DataFrame(training_times)
training_times_df.to_csv(os.path.join(info_dir, 'proximal_policy_optimization_training_times.csv'), index=False, float_format='%.5f')

Saved PPO model
Trained model for 5 warehouses, 400 customers, uneven capacity
